## 🔧 Setup

We import required libraries:
- SQLite for database
- OpenAI (OpenRouter compatible)
- Gradio for UI
- dotenv for API key

In [ ]:
import os
import json
import sqlite3
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

MODEL = "openai/gpt-4o-mini"

## 🗄️ Database Setup

We create a SQLite database to store flight prices.
If the table already exists, it will not be recreated.

In [ ]:
conn = sqlite3.connect("flights.db", check_same_thread=False)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS flights (
    city TEXT PRIMARY KEY,
    price INTEGER
)
""")

conn.commit()

## 🛠️ Tool Functions

These are the functions the LLM can call:
- Get price (multi-city)
- Add price (learning capability)

In [ ]:
def get_price(cities):
    results = {}
    for city in cities:
        cursor.execute("SELECT price FROM flights WHERE city = ?", (city,))
        result = cursor.fetchone()

        if result:
            results[city] = result[0]
        else:
            results[city] = "NOT_FOUND"   # 🔥 KEY FIX

    return results


def add_price(city, price):
    if price <= 0:
        return "Invalid price"

    cursor.execute(
        "INSERT OR REPLACE INTO flights (city, price) VALUES (?, ?)",
        (city, price)
    )
    conn.commit()
    return f"Saved {city} → ${price}"

## 🌱 Seed Initial Data

We insert a few default cities and prices.
If they already exist, they will be ignored.

In [ ]:
def seed_data():
    default_data = [
        ("London", 500),
        ("Paris", 450),
        ("New York", 700),
        ("Tokyo", 800),
        ("Dubai", 300)
    ]

    for city, price in default_data:
        cursor.execute(
            "INSERT OR IGNORE INTO flights (city, price) VALUES (?, ?)",
            (city, price)
        )

    conn.commit()
    print("✅ Seed data inserted (if not already present)")


seed_data()

## 🔧 Tool Definitions

We define tools in a format the LLM understands.

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_price",
            "description": "Get flight prices for cities",
            "parameters": {
                "type": "object",
                "properties": {
                    "cities": {
                        "type": "array",
                        "items": {"type": "string"}
                    }
                },
                "required": ["cities"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "add_price",
            "description": "Add or update flight price",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string"},
                    "price": {"type": "integer"}
                },
                "required": ["city", "price"]
            }
        }
    }
]

## 🧠 System Prompt

Defines behavior of the assistant.

In [ ]:
SYSTEM_PROMPT = """
You are a smart airline assistant.

Rules:
1. If user mentions a single city → assume they want price for that city.
2. Always call get_price with that city.
3. If NOT_FOUND → ask user for price.
4. If user provides price → call add_price.
5. Always respond in markdown.
"""

## 🔁 Agent Loop

Handles:
- Tool calling
- Multi-step reasoning
- Iterative responses

In [ ]:
def chat_with_agent(user_input, history=[]):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    messages.extend(history)
    messages.append({"role": "user", "content": user_input})

    while True:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )

        msg = response.choices[0].message

        # ✅ ONLY if tool is requested
        if msg.tool_calls:

            messages.append(msg)  # 🔥 IMPORTANT (assistant message with tool_calls)

            for tool_call in msg.tool_calls:
                name = tool_call.function.name
                args = json.loads(tool_call.function.arguments)

                if name == "get_price":
                    result = get_price(**args)

                elif name == "add_price":
                    result = add_price(**args)

                # ✅ tool response MUST follow tool_call
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(result)
                })

            continue  # loop again

        else:
            messages.append(msg)
            return msg.content, messages

## 🌐 Gradio Interface

Creates a chat UI for interacting with the agent.

In [ ]:
chat_history = []

def respond(message, history):
    try:
        response, updated_messages = chat_with_agent(message, history)
        return response
    except Exception as e:
        return f"⚠️ Error occurred: {str(e)}"

demo = gr.ChatInterface(
    fn=respond,
    title="✈️ Airline AI Assistant",
    description="Ask for flight prices or teach new ones!"
)

demo.launch()

In [ ]:
cursor.execute("SELECT * FROM flights")
print(cursor.fetchall())